In [5]:
import pandas as pd
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

print("=== MEMULAI FASE 1: PREDICTIVE BASELINE ===")

os.makedirs('models', exist_ok=True)

# ==========================================
# LANGKAH 1 & 2: Load Data & Pisahkan Target
# ==========================================
df = pd.read_csv('university_student_stress_dataset.csv')

# Drop Stress_Score untuk mencegah target leakage
X = df.drop(columns=['Stress_Level', 'Stress_Score'])
y = df['Stress_Level']

# ==========================================
# LANGKAH 3: Encoding Fitur Kategorik
# ==========================================
X_encoded = pd.get_dummies(X, drop_first=True)
joblib.dump(X_encoded.columns, 'models/feature_columns.pkl')
print("Nama kolom fitur berhasil disimpan.")

# ==========================================
# LANGKAH 4: Dataset Split 
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# ==========================================
# LANGKAH 5: Scaling Data & Simpan Scaler
# ==========================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, 'models/scaler.pkl')
print("Scaler berhasil diekspor ke 'models/scaler.pkl'.")

# ==========================================
# LANGKAH 6: Pemodelan & Simpan Model
# ==========================================
model_lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model_lr.fit(X_train_scaled, y_train)

joblib.dump(model_lr, 'models/baseline_lr_model.pkl')
print("[Model Regresi Logistik berhasil diekspor ke 'models/baseline_lr_model.pkl'.")

# ==========================================
# LANGKAH 7: Evaluasi Metrik Lengkap
# ==========================================
y_pred = model_lr.predict(X_test_scaled)
y_pred_proba = model_lr.predict_proba(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted')

print("\n=== HASIL EVALUASI METRIK KESELURUHAN ===")
print(f"Overall Accuracy : {accuracy:.4f}")
print(f"Precision        : {precision:.4f}")
print(f"Recall           : {recall:.4f}")
print(f"F1-Score         : {f1:.4f}")
print(f"AUC (ROC)        : {auc:.4f}\n")

print("=== DETAIL CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

=== MEMULAI FASE 1: PREDICTIVE BASELINE ===
Nama kolom fitur berhasil disimpan.
Scaler berhasil diekspor ke 'models/scaler.pkl'.
[Model Regresi Logistik berhasil diekspor ke 'models/baseline_lr_model.pkl'.

=== HASIL EVALUASI METRIK KESELURUHAN ===
Overall Accuracy : 0.9883
Precision        : 0.9895
Recall           : 0.9883
F1-Score         : 0.9886
AUC (ROC)        : 1.0000

=== DETAIL CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

        High       0.90      1.00      0.95        65
         Low       1.00      1.00      1.00       249
      Medium       1.00      0.98      0.99       286

    accuracy                           0.99       600
   macro avg       0.97      0.99      0.98       600
weighted avg       0.99      0.99      0.99       600



In [6]:
# ==========================================
# BONUS: CEK FITUR PALING BERPENGARUH
# ==========================================
import numpy as np

print("\n=== BOBOT FITUR (COEFFICIENTS) UNTUK KELAS HIGH ===")
# Mengambil koefisien untuk kelas 'High' (indeks 0 secara alfabetis: High, Low, Medium)
coefficients = model_lr.coef_[0]
feature_importance = pd.DataFrame({
    'Fitur': X_encoded.columns,
    'Bobot': coefficients
}).sort_values(by='Bobot', ascending=False)

print(feature_importance)



=== BOBOT FITUR (COEFFICIENTS) UNTUK KELAS HIGH ===
                                 Fitur     Bobot
7                          Screen_Time  4.891124
4                      Assignment_Load  4.059795
10                       Anxiety_Level  3.985831
3                       Exam_Frequency  3.921917
6                     Social_Media_Use  3.596401
16  University_Type_Private University  0.136668
12                         Tuition_Yes  0.077391
2                     Class_Attendance  0.064447
11                         Gender_Male  0.001163
1                          Study_Hours -0.006473
17   University_Type_Public University -0.014416
14             Family_Income_Level_Low -0.020927
8                        Peer_Pressure -0.047598
0                                  Age -0.054940
15          Family_Income_Level_Medium -0.086510
13               Physical_Exercise_Yes -1.518683
5                          Sleep_Hours -2.449217
9                       Family_Support -4.008400
